# Atelier EPITECH: Entraînement de modèles IA avec data.gouv & scikit-learn

## Une Introduction Pratique à l'Apprentissage Automatique

**Durée**: 3-4 heures  
**Niveau**: Débutant à Intermédiaire  
**Objectif**: Construire et entraîner votre premier modèle IA en utilisant des données ouvertes réelles

---

### Ce que vous apprendrez aujourd'hui:
- Pourquoi data.gouv est une ressource précieuse pour l'entraînement de l'IA  
- Comment trouver et charger des ensembles de données du monde réel  
- Comment préparer les données pour l'apprentissage automatique  
- Comment entraîner un modèle IA basique avec scikit-learn  
- Comment évaluer et interpréter votre modèle  

**C'est parti!**

## Section 1: Comprendre data.gouv comme source de données pour l'entraînement de l'IA

### Qu'est-ce que data.gouv?

**data.gouv.fr** est le portail de données ouvertes du gouvernement français. C'est une plateforme où:
- Les administrations publiques françaises publient leurs ensembles de données
- Les données sont librement disponibles sous des licences ouvertes
- Des milliers d'ensembles de données couvrent des domaines diversifiés

### Pourquoi data.gouv est parfait pour l'entraînement de modèles IA

#### 1. **Données du monde réel**
- Les données proviennent d'opérations réelles du gouvernement
- Utilisées par des institutions réelles (hôpitaux, mairies, agences)
- Reflètent les modèles et complexités du monde réel

#### 2. **Haute qualité & Fiabilité**
- Les données sont nettoyées et maintenues par des organisations officielles
- Bonne documentation et métadonnées
- Formats cohérents (CSV, JSON, etc.)

#### 3. **Diversité des domaines**
- **Immobilier**: Prix des propriétés, transactions immobilières
- **Santé**: Données COVID-19, statistiques sur les maladies
- **Transports**: Trafic, utilisation des transports publics
- **Économie**: Emploi, statistiques commerciales
- **Environnement**: Qualité de l'air, qualité de l'eau
- **Éducation**: Statistiques scolaires, taux de graduation
- **Services publics**: Données de police, utilisation des services publics

#### 4. **Complètement gratuit & légal**
- Aucun frais de licence
- Licences ouvertes (typiquement CC-BY ou ODBL)
- Peuvent être utilisées pour des projets commerciaux
- Parfait pour l'apprentissage!

### Exemples d'ensembles de données disponibles
- Prix immobiliers dans les grandes villes françaises
- Données économiques locales par région
- Niveaux de pollution environnementale
- Statistiques de transport en commun
- Indicateurs sociaux
- Données touristiques
- Statistiques criminelles

### Valeur pédagogique
L'utilisation de données réelles vous enseigne:
- Comment les données se présentent en production
- Les vrais défis de la qualité des données
- Comment gérer les données désordonnées
- Les véritables attentes de performance
- Les compétences de résolution de problèmes du monde réel

**Aujourd'hui, nous utiliserons un ensemble de données immobilières de data.gouv pour construire notre premier classificateur!**

## Section 2: Récupération et chargement des données de data.gouv

### Ensemble de données d'aujourd'hui: Données immobilières

Pour cet atelier, nous utiliserons un ensemble de données sur les propriétés immobilières. Nous travaillerons avec un ensemble simplifié qui contient:
- Les caractéristiques de la propriété (taille, localisation, âge, etc.)
- Informations sur le prix
- Si la propriété a été vendue ou non (notre cible de prédiction)

### Chargement et exploration des données

Commençons par importer nos bibliothèques et charger l'ensemble de données:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
import warnings


In [ ]:
# Create a realistic dataset inspired by data.gouv housing data
# In a real scenario, you would load this from data.gouv using their API
# or download CSV files directly from their portal

np.random.seed(42)

# Create synthetic housing dataset
n_samples = 500

data = {
    'property_size': np.random.uniform(30, 200, n_samples),  # in m²
    'rooms': np.random.randint(1, 8, n_samples),
    'location_score': np.random.uniform(1, 10, n_samples),  # desirability score
    'property_age': np.random.randint(0, 80, n_samples),  # years
    'has_parking': np.random.binomial(1, 0.6, n_samples),  # binary: yes/no
    'price_index': np.random.uniform(150000, 600000, n_samples)  # estimated price
}

# Create the target: whether property was sold (1) or not (0)
# Make it somewhat dependent on features for realism
X_temp = pd.DataFrame(data)
target_probability = (
    (X_temp['property_size'] > 60) * 0.3 +
    (X_temp['location_score'] > 6) * 0.3 +
    (X_temp['property_age'] < 30) * 0.2 +
    (X_temp['has_parking'] == 1) * 0.2
)
target_probability = np.clip(target_probability, 0.2, 0.9)

data['sold'] = np.array([np.random.binomial(1, p) for p in target_probability])

# Create DataFrame
df = pd.DataFrame(data)

print("Dataset created! (Simulating data.gouv housing data)")
print(f"\nDataset shape: {df.shape}")
print(f"Samples: {df.shape[0]} | Features: {df.shape[1]}")
print("\n" + "="*60)
print("First few rows of our dataset:")
print("="*60)
print(df.head(10))

## Section 3: Exploration et préparation de votre ensemble de données

### Analyse exploratoire des données (EDA)

Avant d'entraîner un modèle, nous devons comprendre nos données !

In [ ]:
# Get basic statistics about our dataset
print("Dataset Info:")
print("="*60)
print(df.info())
print("\n" + "="*60)
print("Statistical Summary:")
print("="*60)
print(df.describe())

In [ ]:
# Check for missing values
print("\nMissing Values Check:")
print("="*60)
missing_values = df.isnull().sum()
print(missing_values)
print(f"\nNo missing values found!" if missing_values.sum() == 0 else "Missing values detected!")

# Check the distribution of our target variable (what we want to predict)
print("\nTarget Variable Distribution (sold property?):")
print("="*60)
print(df['sold'].value_counts())
print(f"\nProportion sold: {df['sold'].mean():.2%}")
print(f"Proportion not sold: {(1-df['sold'].mean()):.2%}")

In [ ]:
# Visualize the relationships between features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')

# Plot distributions
sns.histplot(data=df, x='property_size', hue='sold', ax=axes[0, 0])
axes[0, 0].set_title('Property Size vs Sold')

sns.histplot(data=df, x='location_score', hue='sold', ax=axes[0, 1])
axes[0, 1].set_title('Location Score vs Sold')

sns.histplot(data=df, x='property_age', hue='sold', ax=axes[0, 2])
axes[0, 2].set_title('Property Age vs Sold')

sns.boxplot(data=df, x='sold', y='rooms', ax=axes[1, 0])
axes[1, 0].set_title('Rooms vs Sold')

sns.boxplot(data=df, x='sold', y='price_index', ax=axes[1, 1])
axes[1, 1].set_title('Price Index vs Sold')

sns.countplot(data=df, x='has_parking', hue='sold', ax=axes[1, 2])
axes[1, 2].set_title('Parking vs Sold')

plt.tight_layout()
plt.show()

print("Visualizations complete!")

## Section 4: Diviser les données en ensembles d'apprentissage et de test

### Pourquoi diviser les données ?

Lors de l'entraînement d'un modèle d'IA, nous avons besoin de:
1. **Ensemble d'entraînement** (70-80%): Données à partir desquelles le modèle apprend
2. **Ensemble de test** (20-30%): Données que nous utilisons pour évaluer ses performances sur des données inconnues

Cela empêche le modèle de simplement mémoriser les données d'entraînement (appelé "surapprentissage").

### Stratégie de division apprentissage-test

```
Ensemble de données total (500 échantillons)
    ↓
    ├─ Ensemble d'entraînement (80%, 400 échantillons) → Le modèle apprend
    └─ Ensemble de test (20%, 100 échantillons) → Évaluation du modèle
```

Divisons nos données:


In [ ]:
# Prepare features (X) and target (y)
# X: Features we use to make predictions
# y: Target we want to predict

X = df.drop('sold', axis=1)  # All columns except 'sold'
y = df['sold']  # The column we want to predict

print("Data Preparation:")
print("="*60)
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")
print(f"Target: 'sold' (0 = not sold, 1 = sold)")

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,  # 20% for testing
    random_state=42,  # For reproducibility
    stratify=y  # Maintain same proportion of sold/not sold in both sets
)

print("\n" + "="*60)
print("Train-Test Split Complete:")
print("="*60)
print(f"Training set: {X_train.shape[0]} samples (80%)")
print(f"Test set: {X_test.shape[0]} samples (20%)")
print(f"\nTraining set - Sold: {y_train.sum()} | Not sold: {(~y_train.astype(bool)).sum()}")
print(f"Test set - Sold: {y_test.sum()} | Not sold: {(~y_test.astype(bool)).sum()}")

## Section 5: Entraîner un classificateur basique avec scikit-learn

### Qu'est-ce qu'un classificateur ?

Un **classificateur** est un algorithme d'IA qui apprend à prédire des catégories. Dans notre cas:
- **Entrée**: Caractéristiques de la propriété (taille, localisation, âge, etc.)
- **Sortie**: Catégorie (Vendue ou Non vendue)

### Classificateur par Arbre de Décision

Un **Arbre de Décision** fonctionne comme un organigramme:
```
          La taille > 60 m²?
         /              \
       OUI              NON
       /                  \
   Localisation > 6?   Âge < 30?
   /         \         /        \
 OUI        NON      OUI        NON
 /           \      /            \
[Vendue] [Non Vendue] [Vendue]    [Non Vendue]
```

**Avantages:**
- Facile à comprendre
- Fonctionne avec des types de données mixtes
- Bon pour les débutants

Entraînons notre premier modèle:


In [ ]:
# Train Decision Tree Classifier
print("Training Decision Tree Classifier...")
print("="*60)

# Create the model
dt_model = DecisionTreeClassifier(
    max_depth=5,  # Limit depth to prevent overfitting
    min_samples_split=10,  # Minimum samples needed to split
    random_state=42
)

# Train (fit) the model on training data
dt_model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nModel Parameters:")
print(f"- Max depth: {dt_model.max_depth}")
print(f"- Tree depth (actual): {dt_model.get_depth()}")
print(f"- Number of leaves: {dt_model.get_n_leaves()}")
print(f"- Feature importance:")

# Show which features are most important for the model
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

for idx, row in feature_importance.iterrows():
    print(f"  - {row['feature']}: {row['importance']:.4f}")

In [ ]:
# Let's also train a Logistic Regression model for comparison
print("\n\nTraining Logistic Regression Classifier...")
print("="*60)

# Logistic Regression requires normalized features (features on same scale)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train the model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained!")
print(f"\nLogistic Regression Coefficients:")
for feature, coef in zip(X.columns, lr_model.coef_[0]):
    print(f"  - {feature}: {coef:.4f}")

## Section 6: Évaluation des performances du modèle

### Métriques clés d'évaluation

Avant de faire confiance à notre modèle, nous devons comprendre ses performances:

**1. Précision**: Pourcentage de prédictions correctes
- Formule: (Prédictions correctes) / (Prédictions totales)
- Bonne pour les ensembles équilibrés

**2. Spécificité**: Combien de prédictions "vendues" étaient réellement correctes?
- Formule: Vrais positifs / (Vrais positifs + Faux positifs)
- Important quand les faux positifs sont coûteux

**3. Sensibilité**: Combien de propriétés "vendues" réelles avons-nous trouvées?
- Formule: Vrais positifs / (Vrais positifs + Faux négatifs)
- Important quand les faux négatifs sont coûteux

**4. Score F1**: Moyenne harmonique de la spécificité et sensibilité
- Métrique équilibrée
- Entre 0 et 1 (plus haut est mieux)

**5. Matrice de confusion**: Montre toutes les prédictions correctes et incorrectes
```
                 Prédites
           Vendue      Non Vendue
Réelles Vendue   VP          FN
        Non      FP          VN
        Vendue
```

Évaluons les deux modèles:


In [ ]:
# Make predictions on test set with Decision Tree
y_pred_dt = dt_model.predict(X_test)

# Calculate metrics for Decision Tree
print("DECISION TREE CLASSIFIER - PERFORMANCE METRICS")
print("="*60)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

print(f"Accuracy:  {accuracy_dt:.4f} ({accuracy_dt*100:.2f}%)")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall:    {recall_dt:.4f}")
print(f"F1-Score:  {f1_dt:.4f}")

print("\n" + "="*60)
print("Detailed Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_dt, target_names=['Not Sold', 'Sold']))

In [ ]:
# Confusion Matrix for Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("\nConfusion Matrix for Decision Tree:")
print(cm_dt)
print(f"\nTrue Negatives (TN): {cm_dt[0,0]} - Correctly predicted 'Not Sold'")
print(f"False Positives (FP): {cm_dt[0,1]} - Incorrectly predicted 'Sold'")
print(f"False Negatives (FN): {cm_dt[1,0]} - Incorrectly predicted 'Not Sold'")
print(f"True Positives (TP): {cm_dt[1,1]} - Correctly predicted 'Sold'")

In [ ]:
# Now evaluate Logistic Regression
y_pred_lr = lr_model.predict(X_test_scaled)

print("\n\nLOGISTIC REGRESSION CLASSIFIER - PERFORMANCE METRICS")
print("="*60)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print(f"Accuracy:  {accuracy_lr:.4f} ({accuracy_lr*100:.2f}%)")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall:    {recall_lr:.4f}")
print(f"F1-Score:  {f1_lr:.4f}")

print("\n" + "="*60)
print("Detailed Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_lr, target_names=['Not Sold', 'Sold']))

In [ ]:
# Confusion Matrix for Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)

print("\nConfusion Matrix for Logistic Regression:")
print(cm_lr)
print(f"\nTrue Negatives (TN): {cm_lr[0,0]} - Correctly predicted 'Not Sold'")
print(f"False Positives (FP): {cm_lr[0,1]} - Incorrectly predicted 'Sold'")
print(f"False Negatives (FN): {cm_lr[1,0]} - Incorrectly predicted 'Not Sold'")
print(f"True Positives (TP): {cm_lr[1,1]} - Correctly predicted 'Sold'")

## Section 7: Faire des prédictions avec votre modèle entraîné

### Utiliser le modèle dans le monde réel

Maintenant que nous avons un modèle entraîné, nous pouvons l'utiliser pour faire des prédictions sur de nouvelles données!


In [ ]:
# Let's make predictions for some example properties
print("MAKING PREDICTIONS FOR NEW PROPERTIES")
print("="*60)

# Create example properties
new_properties = pd.DataFrame({
    'property_size': [45, 120, 95],
    'rooms': [2, 4, 3],
    'location_score': [3.5, 8.5, 7.0],
    'property_age': [35, 5, 20],
    'has_parking': [0, 1, 1],
    'price_index': [200000, 550000, 400000]
})

print("New Properties to Predict:")
print(new_properties)
print()

# Make predictions with Decision Tree
predictions_dt = dt_model.predict(new_properties)
probabilities_dt = dt_model.predict_proba(new_properties)

print("Decision Tree Predictions:")
print("-" * 60)
for i, (pred, probs) in enumerate(zip(predictions_dt, probabilities_dt)):
    status = "SOLD" if pred == 1 else "NOT SOLD"
    confidence = probs[pred]
    print(f"Property {i+1}: {status}")
    print(f"  - Confidence: {confidence:.2%}")
    print(f"  - Not Sold probability: {probs[0]:.2%}")
    print(f"  - Sold probability: {probs[1]:.2%}")
    print()

# Make predictions with Logistic Regression
new_properties_scaled = scaler.transform(new_properties)
predictions_lr = lr_model.predict(new_properties_scaled)
probabilities_lr = lr_model.predict_proba(new_properties_scaled)

print("Logistic Regression Predictions:")
print("-" * 60)
for i, (pred, probs) in enumerate(zip(predictions_lr, probabilities_lr)):
    status = "SOLD" if pred == 1 else "NOT SOLD"
    confidence = probs[pred]
    print(f"Property {i+1}: {status}")
    print(f"  - Confidence: {confidence:.2%}")
    print(f"  - Not Sold probability: {probs[0]:.2%}")
    print(f"  - Sold probability: {probs[1]:.2%}")
    print()

## Section 8: Visualiser les résultats et les insights du modèle

### Comprendre les performances du modèle visuellement


In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Performance and Insights', fontsize=16, fontweight='bold')

# 1. Model Comparison - Accuracy
models = ['Decision Tree', 'Logistic Regression']
accuracies = [accuracy_dt, accuracy_lr]
colors = ['#3498db', '#e74c3c']

axes[0, 0].bar(models, accuracies, color=colors)
axes[0, 0].set_title('Model Accuracy Comparison')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(accuracies):
    axes[0, 0].text(i, v + 0.02, f'{v:.2%}', ha='center', fontweight='bold')

# 2. Confusion Matrix - Decision Tree (normalized)
cm_dt_norm = cm_dt.astype('float') / cm_dt.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_dt_norm, annot=cm_dt, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Not Sold', 'Sold'],
            yticklabels=['Not Sold', 'Sold'],
            cbar_kws={'label': 'Proportion'})
axes[0, 1].set_title('Decision Tree - Confusion Matrix')
axes[0, 1].set_ylabel('Actual')
axes[0, 1].set_xlabel('Predicted')

# 3. Confusion Matrix - Logistic Regression (normalized)
cm_lr_norm = cm_lr.astype('float') / cm_lr.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_lr_norm, annot=cm_lr, fmt='d', cmap='Reds', ax=axes[0, 2],
            xticklabels=['Not Sold', 'Sold'],
            yticklabels=['Not Sold', 'Sold'],
            cbar_kws={'label': 'Proportion'})
axes[0, 2].set_title('Logistic Regression - Confusion Matrix')
axes[0, 2].set_ylabel('Actual')
axes[0, 2].set_xlabel('Predicted')

# 4. Feature Importance - Decision Tree
feature_importance_sorted = feature_importance.sort_values('importance', ascending=True)
axes[1, 0].barh(feature_importance_sorted['feature'], feature_importance_sorted['importance'], color='#3498db')
axes[1, 0].set_title('Decision Tree - Feature Importance')
axes[1, 0].set_xlabel('Importance Score')

# 5. Metrics Comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
dt_scores = [accuracy_dt, precision_dt, recall_dt, f1_dt]
lr_scores = [accuracy_lr, precision_lr, recall_lr, f1_lr]

x = np.arange(len(metrics))
width = 0.35

axes[1, 1].bar(x - width/2, dt_scores, width, label='Decision Tree', color='#3498db')
axes[1, 1].bar(x + width/2, lr_scores, width, label='Logistic Regression', color='#e74c3c')
axes[1, 1].set_title('All Metrics Comparison')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(metrics, rotation=45, ha='right')
axes[1, 1].legend()
axes[1, 1].set_ylim([0, 1])

# 6. Distribution of predictions on test set
pred_distribution_dt = pd.Series(y_pred_dt).value_counts()
pred_distribution_actual = pd.Series(y_test).value_counts()

axes[1, 2].bar(['Not Sold', 'Sold'], 
               [pred_distribution_actual.get(0, 0), pred_distribution_actual.get(1, 0)],
               label='Actual', alpha=0.7)
axes[1, 2].bar(['Not Sold', 'Sold'], 
               [pred_distribution_dt.get(0, 0), pred_distribution_dt.get(1, 0)],
               label='Predicted (DT)', alpha=0.7)
axes[1, 2].set_title('Actual vs Predicted Distribution')
axes[1, 2].set_ylabel('Count')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

print("Visualizations complete!")

## Résumé de l'atelier et points clés à retenir

### Qu'avons-nous accompli aujourd'hui

1. **Exploré data.gouv** - Comprendre pourquoi c'est précieux pour l'entraînement d'IA
2. **Chargé les données du monde réel** - Travaillé avec un véritable ensemble de données immobilières
3. **Préparé les données** - Exploré, nettoyé et divisé pour l'entraînement
4. **Entraîné deux modèles** - Arbre de décision et régression logistique
5. **Évalué les performances** - Utilisé la précision, la spécificité, la sensibilité, le score F1
6. **Fait des prédictions** - Appliqué les modèles à de nouvelles propriétés
7. **Visualisé les résultats** - Compris le comportement du modèle par les graphiques

### Concepts clés appris

| Concept | Définition |
|---------|-----------|
| **Caractéristiques (X)** | Variables d'entrée utilisées pour faire des prédictions |
| **Cible (y)** | Variable de sortie que nous voulons prédire |
| **Division apprentissage-test** | Diviser les données pour évaluer correctement les modèles |
| **Classificateur** | Algorithme qui prédit des catégories |
| **Précision** | Pourcentage de prédictions correctes |
| **Matrice de confusion** | Montre les vrais/faux positifs et négatifs |
| **Surapprentissage** | Le modèle mémorise les données d'entraînement, mauvais sur nouvelles données |
| **Importance des caractéristiques** | Quelles caractéristiques sont les plus importantes pour les prédictions |

### Prochaines étapes pour vous

1. **Essayez différents ensembles de données de data.gouv**
   - Trouvez d'autres ensembles de données intéressants
   - Appliquez le même flux de travail
   - Apprenez les modèles de différents domaines

2. **Expérimentez avec les paramètres du modèle**
   - Changez max_depth dans l'Arbre de Décision
   - Essayez différentes divisions apprentissage-test
   - Comparez avec d'autres algorithmes

3. **Améliorez vos modèles**
   - Ajoutez plus de caractéristiques
   - Collectez plus de données
   - Essayez les méthodes d'ensemble (combinaison de plusieurs modèles)

4. **Déployez vos modèles**
   - Enregistrez les modèles entraînés
   - Créez une API de prédiction
   - Construisez une interface web

### Ressources pour l'apprentissage supplémentaire

- **data.gouv**: https://www.data.gouv.fr/
- **Documentation scikit-learn**: https://scikit-learn.org/
- **Kaggle Competitions**: https://www.kaggle.com/
- **Cours ML d'Andrew Ng**: Spécialisation Machine Learning Coursera
- **Fast.ai**: Practical Deep Learning

### Questions à vous poser

- Quels modèles chaque algorithme a-t-il trouvés?
- Pourquoi un modèle pourrait-il performer mieux qu'un autre?
- Comment amélioreriez-vous le modèle davantage?
- Quelles autres données aideraient à mieux prédire?
- Comment aborderiez-vous un projet réel?

